# SGLang 批量推理压测 — 简易脚本实现

**定位：** 使用简易 Python 脚本对 SGLang 进行批量推理与并发压测，对比不同并发度下的吞吐量和延迟表现。

> **一条命令启动服务：**

```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
```

*本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台*

---

## 硬件与环境要求

| 项目 | 最低要求 | 推荐配置 |
|------|---------|----------|
| GPU | NVIDIA（支持 CUDA） | RTX 3060 及以上 |
| 显存 (VRAM) | ≥ 6 GB | ≥ 8 GB |
| 驱动版本 | ≥ 525.0 | 最新稳定版 |
| CUDA 版本 | ≥ 12.1 | 12.4+ |
| Python | 3.9 – 3.12 | 3.10 |
| 操作系统 | Linux / WSL2 | Ubuntu 22.04 |

---

## 依赖安装

推荐使用虚拟环境隔离依赖：

```bash
# 第一步：创建并激活虚拟环境
python -m venv sglang-env
source sglang-env/bin/activate

# 第二步：安装 SGLang 及所需依赖
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

---

## 使用指南

1. 运行「环境检查」单元格，确认 GPU 和 CUDA 可用
2. 如需在 Notebook 中安装依赖，运行「可选安装」代码单元格
3. 运行「启动 SGLang 服务」单元格，等待服务就绪
4. 按顺序运行压测相关单元格，观察不同并发度下的性能
5. 完成后运行「停止服务释放显存」单元格清理资源

---

In [ ]:
import sys  # 导入系统模块
print(f"Python 版本: {sys.version}")  # 打印 Python 版本信息

import torch  # 导入 PyTorch 深度学习框架
print(f"PyTorch 版本: {torch.__version__}")  # 打印 PyTorch 版本
print(f"CUDA 是否可用: {torch.cuda.is_available()}")  # 检查 CUDA 是否可用

if torch.cuda.is_available():  # 如果 CUDA 可用
    print(f"CUDA 版本: {torch.version.cuda}")  # 打印 CUDA 版本号
    print(f"GPU 名称: {torch.cuda.get_device_name(0)}")  # 打印第一张 GPU 名称
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3  # 计算总显存（GB）
    print(f"GPU 显存: {vram_gb:.1f} GB")  # 打印显存大小
else:  # CUDA 不可用
    print("⚠️ 未检测到 GPU，请检查驱动和 CUDA 安装")  # 提示用户检查环境

In [ ]:
import subprocess  # 导入子进程模块
import sys  # 导入系统模块

subprocess.check_call([  # 执行 pip 安装命令
    sys.executable, "-m", "pip", "install", "-q",  # 使用当前 Python 解释器静默安装
    "sglang[all]", "openai>=1.0.0", "requests>=2.28.0"  # 安装 SGLang 及所需依赖
])  # 安装命令执行完成
print("✅ 依赖安装完成")  # 打印安装成功提示

## 批量推理与吞吐量

在实际生产环境中，推理服务需要同时处理大量请求。衡量推理服务性能的核心指标包括：

- **吞吐量（Throughput）**：单位时间内处理的请求数（requests/sec）
- **延迟（Latency）**：单个请求从发送到收到回复的时间
- **并发能力**：服务同时处理多个请求的能力

SGLang 通过 RadixAttention 等优化技术，在高并发场景下表现优异。下面我们通过简单的脚本来验证。

## 启动 SGLang 服务

在后台启动 SGLang 推理服务，并等待服务就绪。

In [ ]:
import subprocess  # 导入子进程模块
import time  # 导入时间模块
import requests  # 导入 HTTP 请求库
import sys  # 导入系统模块

server_process = subprocess.Popen(  # 以子进程方式启动 SGLang 推理服务
    [  # 命令参数列表
        sys.executable, "-m", "sglang.launch_server",  # 调用 SGLang 服务启动模块
        "--model-path", "Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型路径（首次自动下载）
        "--host", "127.0.0.1",  # 监听本机回环地址
        "--port", "30000",  # 指定服务端口号
    ],  # 命令参数结束
    stdout=subprocess.PIPE,  # 捕获标准输出
    stderr=subprocess.PIPE,  # 捕获标准错误输出
)  # 子进程创建完成
print(f"SGLang 服务进程已启动，PID = {server_process.pid}")  # 打印进程 ID

max_wait = 600  # 最大等待时间 600 秒（含模型下载时间）
start_time = time.time()  # 记录开始时间
ready = False  # 服务就绪标志初始化为 False

while time.time() - start_time < max_wait:  # 在超时范围内持续轮询
    try:  # 尝试连接服务
        r = requests.get("http://127.0.0.1:30000/v1/models", timeout=5)  # 向模型列表接口发请求
        if r.status_code == 200:  # 返回 200 表示服务已就绪
            print(f"✅ SGLang 服务就绪！耗时 {time.time() - start_time:.1f} 秒")  # 打印就绪信息
            ready = True  # 更新就绪标志
            break  # 退出轮询循环
    except requests.ConnectionError:  # 连接失败说明服务尚未启动完成
        pass  # 忽略异常，继续等待
    time.sleep(2)  # 每隔 2 秒重试一次

if not ready:  # 如果超时仍未就绪
    print("❌ 服务启动超时，请检查 GPU 显存或查看进程日志")  # 打印超时错误信息

## 准备测试数据

创建 50 条多样化的中文提示语作为压测数据，并定义不同的并发级别。

In [ ]:
test_prompts = [  # 定义 50 条测试提示语
    "什么是人工智能？",  # 提示 1：AI 基础概念
    "解释一下量子计算的基本原理",  # 提示 2：量子计算
    "Python 和 Java 哪个更适合初学者？",  # 提示 3：编程语言对比
    "什么是区块链技术？",  # 提示 4：区块链
    "如何提高代码质量？",  # 提示 5：软件工程
    "解释一下机器学习中的过拟合",  # 提示 6：机器学习
    "什么是云计算？有哪些优势？",  # 提示 7：云计算
    "TCP 和 UDP 的区别是什么？",  # 提示 8：网络协议
    "什么是 RESTful API？",  # 提示 9：API 设计
    "如何防止 SQL 注入攻击？",  # 提示 10：安全
    "解释一下 Docker 容器技术",  # 提示 11：容器化
    "什么是微服务架构？",  # 提示 12：架构设计
    "Git 的常用命令有哪些？",  # 提示 13：版本控制
    "什么是 NoSQL 数据库？",  # 提示 14：数据库
    "解释一下 OAuth 2.0 认证流程",  # 提示 15：认证授权
    "什么是持续集成/持续部署？",  # 提示 16：CI/CD
    "如何优化网页加载速度？",  # 提示 17：前端优化
    "什么是设计模式？举几个例子",  # 提示 18：设计模式
    "解释一下 GPU 并行计算",  # 提示 19：GPU 计算
    "什么是 Transformer 模型？",  # 提示 20：深度学习
    "如何进行代码审查？",  # 提示 21：开发流程
    "什么是大语言模型？",  # 提示 22：LLM
    "解释一下分布式系统的 CAP 定理",  # 提示 23：分布式系统
    "什么是 WebSocket？",  # 提示 24：实时通信
    "如何选择合适的数据库？",  # 提示 25：技术选型
    "什么是函数式编程？",  # 提示 26：编程范式
    "解释一下 Kubernetes 的作用",  # 提示 27：容器编排
    "什么是 A/B 测试？",  # 提示 28：实验方法
    "如何处理高并发请求？",  # 提示 29：系统设计
    "什么是数据湖？",  # 提示 30：大数据
    "解释一下注意力机制",  # 提示 31：深度学习
    "什么是边缘计算？",  # 提示 32：计算架构
    "如何保证数据安全？",  # 提示 33：数据安全
    "什么是 GraphQL？",  # 提示 34：API 查询语言
    "解释一下 Redis 的使用场景",  # 提示 35：缓存技术
    "什么是 DevOps？",  # 提示 36：运维理念
    "如何进行性能调优？",  # 提示 37：性能优化
    "什么是 Serverless 架构？",  # 提示 38：无服务器
    "解释一下 RPC 远程过程调用",  # 提示 39：分布式通信
    "什么是数据仓库？",  # 提示 40：数据存储
    "如何做好技术文档？",  # 提示 41：文档编写
    "什么是消息队列？",  # 提示 42：异步通信
    "解释一下 CORS 跨域问题",  # 提示 43：Web 安全
    "什么是负载均衡？",  # 提示 44：系统架构
    "如何选择开源协议？",  # 提示 45：开源
    "什么是强化学习？",  # 提示 46：机器学习
    "解释一下索引在数据库中的作用",  # 提示 47：数据库优化
    "什么是 API 网关？",  # 提示 48：微服务
    "如何进行压力测试？",  # 提示 49：测试
    "什么是联邦学习？",  # 提示 50：隐私计算
]  # 测试提示语列表定义完成

concurrency_levels = [1, 5, 10, 20]  # 定义要测试的并发级别列表

print(f"✅ 测试数据准备完成：{len(test_prompts)} 条提示语")  # 打印提示语数量
print(f"📊 待测试并发级别：{concurrency_levels}")  # 打印并发级别

## 同步逐条推理（基线）

先以串行方式逐条发送请求，作为性能基线。

In [ ]:
from openai import OpenAI  # 导入 OpenAI 客户端库
import time  # 导入时间模块

client = OpenAI(  # 创建 OpenAI 兼容客户端实例
    base_url="http://127.0.0.1:30000/v1",  # 指向本地 SGLang 服务地址
    api_key="EMPTY",  # 本地服务无需真实 API Key
)  # 客户端创建完成


def single_request(prompt):  # 定义单条推理请求函数
    """发送单条聊天补全请求并返回结果和耗时"""
    start = time.time()  # 记录请求开始时间
    response = client.chat.completions.create(  # 发送聊天补全请求
        model="Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型名称
        messages=[  # 构造消息列表
            {"role": "system", "content": "你是一个有帮助的AI助手，请简洁回答。"},  # 系统提示
            {"role": "user", "content": prompt},  # 用户提示
        ],  # 消息列表结束
        max_tokens=128,  # 限制回复长度以加快测试速度
    )  # 请求完成
    elapsed = time.time() - start  # 计算请求耗时
    reply = response.choices[0].message.content  # 提取回复文本
    return reply, elapsed  # 返回回复内容和耗时


print("🔄 开始同步逐条推理（基线测试）...")  # 打印开始提示
sequential_latencies = []  # 存储每条请求的延迟
seq_start = time.time()  # 记录整体开始时间

for i, prompt in enumerate(test_prompts):  # 遍历所有测试提示语
    reply, latency = single_request(prompt)  # 发送单条请求
    sequential_latencies.append(latency)  # 记录延迟
    if (i + 1) % 10 == 0:  # 每 10 条打印一次进度
        print(f"  已完成 {i + 1}/{len(test_prompts)} 条")  # 打印进度信息

seq_total_time = time.time() - seq_start  # 计算总耗时
seq_avg_latency = sum(sequential_latencies) / len(sequential_latencies)  # 计算平均延迟
seq_throughput = len(test_prompts) / seq_total_time  # 计算吞吐量

print(f"\n📊 同步基线结果：")  # 打印结果标题
print(f"  总耗时: {seq_total_time:.2f} 秒")  # 打印总耗时
print(f"  平均延迟: {seq_avg_latency:.3f} 秒/请求")  # 打印平均延迟
print(f"  吞吐量: {seq_throughput:.2f} 请求/秒")  # 打印吞吐量

## 并发推理函数

使用 `concurrent.futures.ThreadPoolExecutor` 实现可配置并发度的批量推理。

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed  # 导入线程池和完成迭代器
import time  # 导入时间模块


def concurrent_benchmark(prompts, concurrency):  # 定义并发压测函数
    """以指定并发度发送请求，返回性能指标字典"""
    latencies = []  # 存储每条请求的延迟
    errors = 0  # 错误计数器
    total_start = time.time()  # 记录整体开始时间

    with ThreadPoolExecutor(max_workers=concurrency) as executor:  # 创建指定并发度的线程池
        futures = {  # 提交所有请求任务
            executor.submit(single_request, prompt): prompt  # 将每个提示语提交为异步任务
            for prompt in prompts  # 遍历所有提示语
        }  # 任务提交完成

        for future in as_completed(futures):  # 按完成顺序迭代所有任务
            try:  # 尝试获取任务结果
                reply, latency = future.result(timeout=60)  # 获取结果，超时 60 秒
                latencies.append(latency)  # 记录延迟
            except Exception as e:  # 捕获任何异常
                errors += 1  # 错误计数加一
                print(f"  ⚠️ 请求失败: {e}")  # 打印错误信息

    total_time = time.time() - total_start  # 计算总耗时
    avg_latency = sum(latencies) / len(latencies) if latencies else 0  # 计算平均延迟
    throughput = len(prompts) / total_time  # 计算吞吐量

    return {  # 返回性能指标字典
        "concurrency": concurrency,  # 并发度
        "total_time": total_time,  # 总耗时
        "throughput": throughput,  # 吞吐量（请求/秒）
        "avg_latency": avg_latency,  # 平均延迟（秒）
        "min_latency": min(latencies) if latencies else 0,  # 最小延迟
        "max_latency": max(latencies) if latencies else 0,  # 最大延迟
        "errors": errors,  # 错误数
        "success": len(latencies),  # 成功数
    }  # 字典定义结束


print("✅ 并发推理函数定义完成")  # 打印函数定义成功提示

## 不同并发度压测

分别在并发度 1、5、10、20 下运行完整压测，收集性能数据。

In [ ]:
benchmark_results = []  # 存储所有并发级别的测试结果

for concurrency in concurrency_levels:  # 遍历每个并发级别
    print(f"\n{'=' * 50}")  # 打印分隔线
    print(f"🚀 正在测试并发度 = {concurrency}...")  # 打印当前测试的并发度

    result = concurrent_benchmark(test_prompts, concurrency)  # 执行并发压测
    benchmark_results.append(result)  # 保存测试结果

    print(f"  ✅ 完成 | 总耗时: {result['total_time']:.2f}s")  # 打印总耗时
    print(f"  📈 吞吐量: {result['throughput']:.2f} 请求/秒")  # 打印吞吐量
    print(f"  ⏱️ 平均延迟: {result['avg_latency']:.3f}s")  # 打印平均延迟
    print(f"  📊 延迟范围: {result['min_latency']:.3f}s ~ {result['max_latency']:.3f}s")  # 打印延迟范围
    print(f"  ✅ 成功: {result['success']} | ❌ 失败: {result['errors']}")  # 打印成功和失败数

print(f"\n{'=' * 50}")  # 打印结束分隔线
print("🎉 所有并发级别测试完成！")  # 打印测试完成提示

## 结果汇总与分析

将所有并发级别的测试结果汇总为表格，分析吞吐量随并发度的变化趋势。

In [ ]:
print("\n" + "=" * 80)  # 打印表格上边框
print("📊 SGLang 批量推理压测结果汇总")  # 打印表格标题
print("=" * 80)  # 打印分隔线

header = f"{'并发度':>6} | {'总耗时(s)':>10} | {'吞吐量(req/s)':>14} | {'平均延迟(s)':>11} | {'最小延迟(s)':>11} | {'最大延迟(s)':>11} | {'成功':>4} | {'失败':>4}"  # 定义表头格式
print(header)  # 打印表头
print("-" * 80)  # 打印表头下划线

for r in benchmark_results:  # 遍历所有测试结果
    row = f"{r['concurrency']:>6} | {r['total_time']:>10.2f} | {r['throughput']:>14.2f} | {r['avg_latency']:>11.3f} | {r['min_latency']:>11.3f} | {r['max_latency']:>11.3f} | {r['success']:>4} | {r['errors']:>4}"  # 格式化每行数据
    print(row)  # 打印该行

print("=" * 80)  # 打印表格下边框

print("\n📈 吞吐量变化趋势：")  # 打印趋势分析标题
if len(benchmark_results) >= 2:  # 如果有至少两个数据点
    baseline_tp = benchmark_results[0]["throughput"]  # 取基线吞吐量（并发=1）
    for r in benchmark_results:  # 遍历所有结果
        speedup = r["throughput"] / baseline_tp  # 计算相对基线的加速比
        bar = "█" * int(speedup * 10)  # 生成可视化进度条
        print(f"  并发 {r['concurrency']:>2}: {bar} {r['throughput']:.2f} req/s (加速比: {speedup:.2f}x)")  # 打印吞吐量和加速比

print("\n💡 分析：")  # 打印分析标题
print("  - 吞吐量通常随并发度增加而提升，因为 GPU 可以批量处理多个请求")  # 分析说明 1
print("  - 当 GPU 计算资源饱和时，吞吐量提升会趋于平缓")  # 分析说明 2
print("  - 平均延迟可能随并发度增加而上升，这是正常现象（请求排队等待）")  # 分析说明 3

## 停止服务释放显存

In [ ]:
import os  # 导入操作系统模块
import signal  # 导入信号处理模块

if 'server_process' in dir() and server_process.poll() is None:  # 检查服务进程是否仍在运行
    os.kill(server_process.pid, signal.SIGTERM)  # 向服务进程发送终止信号
    server_process.wait(timeout=10)  # 等待进程退出，最多等 10 秒
    print(f"✅ SGLang 服务已停止 (PID={server_process.pid})")  # 打印停止成功信息
else:  # 服务进程不存在或已结束
    print("ℹ️ 服务进程未运行或已结束")  # 打印提示信息

if 'torch' in dir() and torch.cuda.is_available():  # 检查 PyTorch 和 CUDA 是否可用
    torch.cuda.empty_cache()  # 清空 GPU 显存缓存
    print("✅ GPU 显存缓存已清理")  # 打印显存清理完成信息

## 常见问题排查

### 1. 高并发下请求超时

**现象**：并发度较高时部分请求超时或返回错误。

**原因**：服务端 token 总量不足以支撑高并发请求同时处理。

**解决**：
- 启动服务时增大 `--max-total-tokens` 参数以允许更多并发请求
- 适当降低并发度，避免超出 GPU 处理能力
- 减小每条请求的 `max_tokens`，降低单请求资源占用

### 2. 吞吐量无法继续提升（平台期）

**现象**：增加并发度后吞吐量不再增长甚至下降。

**原因**：GPU 计算资源已饱和，继续增加并发只会增加排队延迟。

**解决**：
- 这是正常现象，说明已达到当前硬件的性能上限
- 如需更高吞吐量，可使用更高端的 GPU 或部署多卡/多机方案
- 可通过 `--mem-fraction-static` 参数调整显存分配比例来微调性能

---

**结语**：本教程通过简易 Python 脚本对 SGLang 进行了批量推理压测。通过对比不同并发度下的吞吐量和延迟数据，可以快速了解推理服务的性能特征，为生产环境的容量规划提供参考。